In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://paulgraham.com"
response = requests.get(f"{base_url}/articles.html")
soup = BeautifulSoup(response.text, "html.parser")

essay_urls = sorted({
    f"{base_url}/{a['href']}"
    for a in soup.find_all("a", href=True)
    if a["href"].endswith(".html") and a["href"] != "articles.html"
})

print(f"Found {len(essay_urls)} essay URLs")
print(essay_urls[:5])

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os, time

load_dotenv()
output_dir = Path("../data/raw")
output_dir.mkdir(parents=True, exist_ok=True)

headers = {"User-Agent": os.getenv("USER_AGENT", "rag-pipeline/1.0")}

for url in essay_urls:
    slug = url.split("/")[-1].replace(".html", "")
    out_path = output_dir / f"{slug}.txt"
    if out_path.exists() and out_path.stat().st_size > 200:
        continue  # skip files that already have content

    resp = requests.get(url, headers=headers)
    page_soup = BeautifulSoup(resp.text, "html.parser")

    # Newer essays: body in <p> tags
    paragraphs = [p.get_text() for p in page_soup.find_all("p") if p.get_text().strip()]
    if paragraphs:
        text = "\n\n".join(paragraphs)
    else:
        # Older essays: body in <font face="verdana"> with no <p> wrappers
        font_tags = page_soup.find_all("font", attrs={"face": "verdana"})
        text = "\n\n".join(tag.get_text() for tag in font_tags if tag.get_text().strip())

    out_path.write_text(text, encoding="utf-8")
    time.sleep(0.2)  # polite delay

print(f"Saved {len(list(output_dir.glob('*.txt')))} essays to {output_dir}")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "../data/raw/",
    glob="**/*.txt",
    loader_cls=TextLoader,
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")

In [ ]:
import statistics

word_counts = [len(doc.page_content.split()) for doc in documents]

print(f"Total essays      : {len(documents)}")
print(f"Words min/max/mean: {min(word_counts)} / {max(word_counts)} / {statistics.mean(word_counts):.0f}")
print("\n--- Sample document ---")
print("Source :", documents[0].metadata["source"])
print("Preview:", documents[0].page_content[:500])